# Extraccion de Datos e Ingenieria de variables

## 1. Exploración y Selección de la Muestra de Datos

Para este proyecto de clustering y análisis táctico, utilizaremos la base de datos abierta de **StatsBomb (StatsBomb Open Data)**. Tras explorar las competencias disponibles mediante la API `statsbombpy`, la decisión estratégica fue centrar el análisis en la **temporada 2015/2016**.

**¿Por qué la temporada 15/16?**
Es el único corte temporal en el repositorio gratuito de StatsBomb que contiene información detallada de las "5 Grandes Ligas" de Europa (Premier League, La Liga, Serie A, 1. Bundesliga y Ligue 1) en un mismo año calendario. Esto nos proporciona un ecosistema de datos simétrico y lo suficientemente masivo (miles de jugadores y eventos) para entrenar modelos de aprendizaje no supervisado sin sesgos regionales.


In [2]:
pip install statsbombpy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 691.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.6 MB/s eta 0:00:00


In [3]:
from statsbombpy import sb
import pandas as pd

# 1. Llamamos a la base de datos que engloba todo
competiciones = sb.competitions()

# 2. Analizamos su composición: ¿Cuántas y cuáles temporadas hay por cada competición?
resumen_base = competiciones.groupby('competition_name').agg(
    Cantidad_Temporadas=('season_name', 'count'),
    Temporadas_Disponibles=('season_name', lambda x: ', '.join(x.astype(str)))
).sort_values(by='Cantidad_Temporadas', ascending=False).reset_index()
resumen_base.columns = ['Competición', 'Cantidad', 'Temporadas Disponibles']

# 3. Visualizamos el Top de competiciones y el detalle de sus años en el Open Data
print("=== COMPETICIONES Y SUS TEMPORADAS EN STATSBOMB ===")
pd.set_option('display.max_colwidth', None)
print(resumen_base.head(15).to_string(index=False))
print("=" * 80)


=== COMPETICIONES Y SUS TEMPORADAS EN STATSBOMB ===
            Competición  Cantidad                                                                                                                                                                               Temporadas Disponibles
       Champions League        18 2018/2019, 2017/2018, 2016/2017, 2015/2016, 2014/2015, 2013/2014, 2012/2013, 2011/2012, 2010/2011, 2009/2010, 2008/2009, 2006/2007, 2004/2005, 2003/2004, 1999/2000, 1972/1973, 1971/1972, 1970/1971
                La Liga        18 2020/2021, 2019/2020, 2018/2019, 2017/2018, 2016/2017, 2015/2016, 2014/2015, 2013/2014, 2012/2013, 2011/2012, 2010/2011, 2009/2010, 2008/2009, 2007/2008, 2006/2007, 2005/2006, 2004/2005, 1973/1974
         FIFA World Cup         8                                                                                                                                                       2022, 2018, 1990, 1986, 1974, 1970, 1962, 1958
FA Women's Super League 

# 2. Estrategia de Extracción y Control de Calidad

Extraer información a nivel de evento para cuatro ligas completas implica manejar bases de datos que superan fácilmente 1.5 GB de peso en memoria, lo que satura rápidamente entornos como Google Colab o Jupyter.

Para solucionar esto, aplicamos un enfoque de **Procesamiento en Lotes (Chunking / MapReduce) directo desde la API**: descargamos un partido, guardamos el resumen estadístico y liberamos la memoria antes de pasar al siguiente partido. Esto evita problemas de RAM y previene la corrupción de tipos de datos (como las coordenadas espaciales) que ocurre al guardar y leer archivos CSV gigantes.

**Filtro de Integridad: La exclusión de la Bundesliga**
Durante la auditoría de los datos, detectamos una anomalía en el repositorio oficial (Issue reportado en la comunidad): el archivo maestro de la 1. Bundesliga 2015/2016 fue sobreescrito y actualmente solo contiene 34 partidos (equivalentes a los encuentros de un solo equipo, el Bayern Múnich), en lugar de los 306 partidos de la temporada completa.

Para evitar un sesgo masivo en el modelo algorítmico, **decidimos eliminar la Bundesliga del ecosistema de datos**. El modelo final se construirá con las 4 ligas restantes (La Liga, Premier League, Serie A y Ligue 1), garantizando que todos los jugadores tengan las mismas oportunidades de sumar minutos y eventos.

In [ ]:
from statsbombpy import sb
import pandas as pd
import warnings
import os
import gc

warnings.filterwarnings('ignore')

ligas_top5 = {
    'La Liga': {'comp_id': 11, 'season_id': 27},
    'Premier League': {'comp_id': 2, 'season_id': 27},
    '1. Bundesliga': {'comp_id': 9, 'season_id': 27},
    'Serie A': {'comp_id': 12, 'season_id': 27},
    'Ligue 1': {'comp_id': 7, 'season_id': 27}
}

nombre_archivo = 'eventos_big5_1516_raw.csv'
# Si el archivo ya existe de un intento fallido anterior, lo borramos para empezar limpio
if os.path.exists(nombre_archivo):
    os.remove(nombre_archivo)

print("=== INICIANDO DESCARGA INCREMENTAL: ECOSISTEMA 'BIG 5' (2015/2016) ===")
print("El archivo se irá escribiendo directamente en el disco para no usar RAM...\n")

primer_chunk = True
lote_eventos = []
contador_partidos = 0
TAMANO_LOTE = 20 # Cada 20 partidos, escribimos en el CSV y vaciamos la RAM

for liga, ids in ligas_top5.items():
    print(f"⚽ Procesando: {liga}...")
    try:
        partidos = sb.matches(competition_id=ids['comp_id'], season_id=ids['season_id'])
        match_ids = partidos['match_id'].unique()

        for i, match_id in enumerate(match_ids, 1):
            try:
                eventos = sb.events(match_id=match_id)
                eventos['competition_name'] = liga
                lote_eventos.append(eventos)
                contador_partidos += 1

                # Cuando llegamos a 20 partidos o al final de la liga, guardamos en el disco
                if contador_partidos % TAMANO_LOTE == 0 or i == len(match_ids):
                    df_chunk = pd.concat(lote_eventos, ignore_index=True)

                    # Si es la primera vez, escribimos con los nombres de las columnas
                    if primer_chunk:
                        df_chunk.to_csv(nombre_archivo, index=False, mode='w')
                        primer_chunk = False
                    # Si ya existe, solo añadimos las filas abajo (modo 'a' de append)
                    else:
                        df_chunk.to_csv(nombre_archivo, index=False, mode='a', header=False)

                    # ¡VACIAR LA RAM!
                    lote_eventos = []
                    del df_chunk
                    gc.collect()

                    print(f"     ✅ {i}/{len(match_ids)} partidos guardados en disco. RAM liberada.")

            except Exception:
                continue
    except Exception as e:
        print(f"  [X] Error en {liga}: {e}")
        continue

print(f"\n✅ ¡Descarga exitosa finalizada! El archivo '{nombre_archivo}' está listo.")

# Forzar la descarga a PC si en Google Colab
try:
    from google.colab import files
    print(f"Iniciando descarga a tu computador...")
    files.download(nombre_archivo)
except ImportError:
    print(f"El archivo está en la misma carpeta de este script.")

=== INICIANDO DESCARGA INCREMENTAL: ECOSISTEMA 'BIG 5' (2015/2016) ===
El archivo se irá escribiendo directamente en el disco para no usar RAM...

⚽ Procesando: La Liga...
     ✅ 20/380 partidos guardados en disco. RAM liberada.
     ✅ 40/380 partidos guardados en disco. RAM liberada.
     ✅ 60/380 partidos guardados en disco. RAM liberada.
     ✅ 80/380 partidos guardados en disco. RAM liberada.
     ✅ 100/380 partidos guardados en disco. RAM liberada.
     ✅ 120/380 partidos guardados en disco. RAM liberada.
     ✅ 140/380 partidos guardados en disco. RAM liberada.
     ✅ 160/380 partidos guardados en disco. RAM liberada.
     ✅ 180/380 partidos guardados en disco. RAM liberada.
     ✅ 200/380 partidos guardados en disco. RAM liberada.
     ✅ 220/380 partidos guardados en disco. RAM liberada.
     ✅ 240/380 partidos guardados en disco. RAM liberada.
     ✅ 260/380 partidos guardados en disco. RAM liberada.
     ✅ 280/380 partidos guardados en disco. RAM liberada.
     ✅ 300/380 parti

### 3. Ingeniería de Variables (Feature Engineering)

StatsBomb entrega más de 120 columnas de datos crudos por evento. Para construir los perfiles tácticos, filtramos a los jugadores con **más de 900 minutos jugados** y consolidamos su información.

La base de datos final exportada se compone de información contextual, detalles de posicionamiento y **16 variables de scouting** estandarizadas por cada 90 minutos (P90) y ajustadas por posesión (PAdj). Estas métricas capturan la esencia del juego moderno:

**Datos de Identificación y Contexto:**
*   **Player:** Nombre completo del jugador.
*   **Team:** Equipo al que pertenece.
*   **Competition Name:** Liga de origen (La Liga, Premier League, Serie A o Ligue 1).
*   **Minutos Jugados:** Tiempo total acumulado en el campo durante la temporada.

**Arquitectura de Posiciones:**
*   **Position Detail:** La posición principal específica en la que el jugador acumuló más minutos durante la temporada (ej. *Left Center Forward*, *Defensive Midfield*).
*   **Position Group:** Categoría macro asignada dinámicamente según la posición específica, agrupando a los jugadores en Portero, Defensa, Mediocampista, Delantero u Otro.

**Métricas Ofensivas y de Finalización:**
*   **xG P90 (Expected Goals):** Calidad y cantidad de las ocasiones de gol generadas.
*   **xG por Tiro:** Eficiencia en la toma de decisiones al momento de rematar (qué tan buenas son las posiciones de tiro).
*   **Asistencias a Tiro P90 (Key Passes):** Pases directos que terminan en un remate de un compañero.

**Métricas de Distribución y Posesión:**
*   **% Centralidad Equipo:** Porcentaje de los pases totales del equipo que pasan por los pies de este jugador. Mide quién es el "director de orquesta".
*   **% Pases de Seguridad:** Proporción de pases hacia atrás o laterales sin ganancia territorial.
*   **% Pases Bajo Presión:** Capacidad técnica para distribuir el balón cuando el rival está realizando una presión física directa.

**Métricas de Progresión y Dinamismo:**
*   **Pases Progresivos P90:** Pases que acercan el balón significativamente hacia el arco rival (mínimo 10 yardas). Vital para evaluar mediocentros creativos.
*   **Pases Último Tercio P90:** Entregas exitosas en la zona de ataque.
*   **Conducciones Progresivas P90 (Carries):** Traslados de balón en los pies que rompen líneas defensivas.
*   **Influencia Global P90:** Cantidad total de intervenciones con el balón.
*   **Faltas Recibidas P90:** Indicador de desequilibrio y protección del balón frente a defensas rivales.
*   **Pérdidas de Balón P90:** Errores no forzados o despojos por parte del rival.

**Métricas Defensivas:**
*   **Intercepciones PAdj P90:** Lectura táctica para cortar líneas de pase, ajustada por la posesión del equipo rival.
*   **Tackles Ganados PAdj P90:** Entradas directas exitosas para recuperar el balón.
*   **Recuperaciones Último Tercio P90:** Robos de balón en zona de ataque, indicador clave para equipos que realizan presión alta (Gegenpressing).
*   **Duelos Aéreos Ganados P90:** Imposición física en balones divididos por alto.

In [6]:
from statsbombpy import sb
import pandas as pd
import numpy as np
import warnings
import gc
import time

warnings.filterwarnings('ignore')

print("🚀 INICIANDO INGENIERÍA DE BIG DATA DIRECTAMENTE DESDE LA API")
start_time = time.time()

# 1. Definimos las 4 Ligas (Excluimos Bundesliga por el error de los 34 partidos)
ligas_top = {
    'La Liga': {'comp_id': 11, 'season_id': 27},
    'Premier League': {'comp_id': 2, 'season_id': 27},
    'Serie A': {'comp_id': 12, 'season_id': 27},
    'Ligue 1': {'comp_id': 7, 'season_id': 27}
}

lista_stats = []
lista_posesion = []
lista_minutos = []

# Función de coordenadas adaptada a la API nativa
def get_x_api(loc):
    if isinstance(loc, list) and len(loc) > 0:
        return float(loc[0])
    return np.nan

total_partidos_procesados = 0

# --- FASE 1: EXTRACCIÓN Y AGREGACIÓN (Partido por Partido) ---
for liga, ids in ligas_top.items():
    print(f"\n⚽ Procesando {liga}...")
    try:
        partidos = sb.matches(competition_id=ids['comp_id'], season_id=ids['season_id'])
        match_ids = partidos['match_id'].unique()

        for i, m_id in enumerate(match_ids, 1):
            try:
                # Descargar partido
                df_eventos = sb.events(match_id=m_id)
                df_players = df_eventos.dropna(subset=['player']).copy()

                if df_players.empty:
                    continue

                # --- NUEVO: Inyectar Liga y Posición desde los eventos ---
                df_players['competition_name'] = liga
                if 'position' in df_players.columns:
                    df_players['position_detail'] = df_players['position'].fillna('Desconocida')
                else:
                    df_players['position_detail'] = 'Desconocida'

                # Extraer Coordenadas
                df_players['start_x'] = df_players.get('location', pd.Series(dtype=object)).apply(get_x_api)
                df_players['pass_end_x'] = df_players.get('pass_end_location', pd.Series(dtype=object)).apply(get_x_api)
                df_players['carry_end_x'] = df_players.get('carry_end_location', pd.Series(dtype=object)).apply(get_x_api)

                # Minutos
                min_match = df_players.groupby(['player'])['minute'].max().reset_index()
                min_match['match_id'] = m_id
                lista_minutos.append(min_match)

                # Posesión del partido
                pases = df_players[df_players['type'] == 'Pass']
                pos_match = pases.groupby('team').size().reset_index(name='pases')
                pos_match['match_id'] = m_id
                lista_posesion.append(pos_match)

                # Máscaras Lógicas Seguras
                df_players['es_pase'] = df_players['type'] == 'Pass'
                df_players['es_pase_progresivo'] = df_players['es_pase'] & ((df_players['pass_end_x'] - df_players['start_x']) >= 10)
                df_players['es_pase_ultimo_tercio'] = df_players['es_pase'] & (df_players['pass_end_x'] >= 80)
                df_players['es_pase_seguridad'] = df_players['es_pase'] & ((df_players['pass_end_x'] - df_players['start_x']) <= 0)

                if 'under_pressure' in df_players.columns:
                    df_players['es_pase_presion'] = df_players['es_pase'] & (df_players['under_pressure'] == True)
                else: df_players['es_pase_presion'] = False

                df_players['es_tiro'] = df_players['type'] == 'Shot'
                df_players['xG'] = df_players.get('shot_statsbomb_xg', pd.Series(0, index=df_players.index)).fillna(0)
                df_players['asistencia_tiro'] = df_players.get('pass_shot_assist', pd.Series(False, index=df_players.index)).fillna(False).astype(bool)

                df_players['es_intercepcion'] = df_players['type'] == 'Interception'
                df_players['es_recuperacion'] = df_players['type'] == 'Ball Recovery'
                df_players['recup_ultimo_tercio'] = df_players['es_recuperacion'] & (df_players['start_x'] >= 80)

                if 'duel_type' in df_players.columns and 'duel_outcome' in df_players.columns:
                    df_players['tackle_ganado'] = (df_players['type'] == 'Duel') & (df_players['duel_type'] == 'Tackle') & (df_players['duel_outcome'].isin(['Won', 'Success In Play']))
                    df_players['duelo_aereo_ganado'] = (df_players['type'] == 'Duel') & (df_players['duel_type'].astype(str).str.contains('Aerial')) & (df_players['duel_outcome'].isin(['Won', 'Success In Play']))
                else:
                    df_players['tackle_ganado'] = False
                    df_players['duelo_aereo_ganado'] = False

                df_players['es_conduccion_prog'] = (df_players['type'] == 'Carry') & ((df_players['carry_end_x'] - df_players['start_x']) >= 10)
                df_players['es_perdida'] = df_players['type'].isin(['Dispossessed', 'Miscontrol'])
                df_players['es_falta_recibida'] = df_players['type'] == 'Foul Won'

                # --- MODIFICADO: Añadimos competition_name y position_detail al groupby ---
                stats_partido = df_players.groupby(['player', 'team', 'competition_name', 'position_detail']).agg(
                    influencia_total=('id', 'count'), pases_totales=('es_pase', 'sum'), pases_progresivos=('es_pase_progresivo', 'sum'),
                    pases_ultimo_tercio=('es_pase_ultimo_tercio', 'sum'), pases_seguridad=('es_pase_seguridad', 'sum'),
                    pases_presion=('es_pase_presion', 'sum'), asistencias_tiro=('asistencia_tiro', 'sum'), tiros_totales=('es_tiro', 'sum'),
                    xG_total=('xG', 'sum'), intercepciones=('es_intercepcion', 'sum'), recuperaciones=('es_recuperacion', 'sum'),
                    recup_ultimo_tercio=('recup_ultimo_tercio', 'sum'), tackles_ganados=('tackle_ganado', 'sum'),
                    duelos_aereos_ganados=('duelo_aereo_ganado', 'sum'), conducciones_prog=('es_conduccion_prog', 'sum'),
                    perdidas_balon=('es_perdida', 'sum'), faltas_recibidas=('es_falta_recibida', 'sum')
                ).reset_index()

                lista_stats.append(stats_partido)
                total_partidos_procesados += 1

                # Reporte en la misma línea
                print(f"  Partidos procesados: {i}/{len(match_ids)}", end='\r')

                # Liberar RAM
                del df_eventos, df_players, pases, stats_partido
                gc.collect()

            except Exception as e:
                continue
    except Exception as e:
        print(f"Error accediendo a la liga {liga}: {e}")

print(f"\n\n✅ Extracción finalizada. Total partidos reales analizados: {total_partidos_procesados}")
print("Consolidando modelo global...")



# --- FASE 2 CORREGIDA: REDUCCIÓN Y CONSOLIDACIÓN ---

# Minutos
df_min = pd.concat(lista_minutos)
minutos_totales = df_min.groupby('player')['minute'].sum().reset_index().rename(columns={'minute': 'minutos_jugados'})

# Posesión
df_pos = pd.concat(lista_posesion)
pases_partido = df_pos.groupby('match_id')['pases'].sum().reset_index(name='totales')
df_pos = pd.merge(df_pos, pases_partido, on='match_id')
df_pos['pos_partido'] = (df_pos['pases'] / df_pos['totales']) * 100
posesion_media = df_pos.groupby('team')['pos_partido'].mean().reset_index(name='posesion_media')

# 🚨 AQUÍ ESTABA EL ERROR: Faltaba volver a declarar estas variables
df_stats_all = pd.concat(lista_stats)
cols_a_sumar = [
    'influencia_total', 'pases_totales', 'pases_progresivos', 'pases_ultimo_tercio',
    'pases_seguridad', 'pases_presion', 'asistencias_tiro', 'tiros_totales', 'xG_total',
    'intercepciones', 'recuperaciones', 'recup_ultimo_tercio', 'tackles_ganados',
    'duelos_aereos_ganados', 'conducciones_prog', 'perdidas_balon', 'faltas_recibidas'
]

# 1. Obtenemos la "Posición Principal" (la que más veces jugó en la temporada)
posiciones_principales = df_stats_all.groupby(['player', 'team', 'competition_name'])['position_detail'].agg(
    lambda x: x.value_counts().index[0]
).reset_index()

# 2. Sumamos TODAS las métricas del jugador, sin importar en cuántas posiciones rotó
df_stats = df_stats_all.groupby(['player', 'team', 'competition_name'])[cols_a_sumar].sum().reset_index()

# 3. Unimos las métricas totales con su posición principal
df_stats = pd.merge(df_stats, posiciones_principales, on=['player', 'team', 'competition_name'])

# Cruce Maestro
perfiles = pd.merge(df_stats, minutos_totales, on='player')
perfiles = pd.merge(perfiles, posesion_media, on='team')


# --- FASE 3: CÁLCULO DE LAS 16 VARIABLES DEFINITIVAS (Filtro >= 900 min) ---
perfiles = perfiles[perfiles['minutos_jugados'] >= 900].copy()
perfiles['factor_padj'] = 50 / (100 - perfiles['posesion_media'])

perfiles['xG P90'] = (perfiles['xG_total'] / perfiles['minutos_jugados']) * 90
perfiles['xG por Tiro'] = np.where(perfiles['tiros_totales'] > 0, perfiles['xG_total'] / perfiles['tiros_totales'], 0)
perfiles['Asistencias a Tiro P90'] = (perfiles['asistencias_tiro'] / perfiles['minutos_jugados']) * 90

pases_equipo_total = perfiles.groupby('team')['pases_totales'].transform('sum')
perfiles['% Centralidad Equipo'] = np.where(pases_equipo_total > 0, (perfiles['pases_totales'] / pases_equipo_total) * 100, 0)

perfiles['Pases Progresivos P90'] = (perfiles['pases_progresivos'] / perfiles['minutos_jugados']) * 90
perfiles['Pases Último Tercio P90'] = (perfiles['pases_ultimo_tercio'] / perfiles['minutos_jugados']) * 90
perfiles['% Pases Seguridad'] = np.where(perfiles['pases_totales'] > 0, (perfiles['pases_seguridad'] / perfiles['pases_totales']) * 100, 0)
perfiles['% Pases Bajo Presión'] = np.where(perfiles['influencia_total'] > 0, (perfiles['pases_presion'] / perfiles['influencia_total']) * 100, 0)

perfiles['Intercepciones PAdj P90'] = ((perfiles['intercepciones'] / perfiles['minutos_jugados']) * 90) * perfiles['factor_padj']
perfiles['Tackles Ganados PAdj P90'] = ((perfiles['tackles_ganados'] / perfiles['minutos_jugados']) * 90) * perfiles['factor_padj']
perfiles['Duelos Aéreos Ganados P90'] = (perfiles['duelos_aereos_ganados'] / perfiles['minutos_jugados']) * 90
perfiles['Recup. Último Tercio P90'] = (perfiles['recup_ultimo_tercio'] / perfiles['minutos_jugados']) * 90

perfiles['Conducciones Progresivas P90'] = (perfiles['conducciones_prog'] / perfiles['minutos_jugados']) * 90
perfiles['Influencia Global P90'] = (perfiles['influencia_total'] / perfiles['minutos_jugados']) * 90
perfiles['Faltas Recibidas P90'] = (perfiles['faltas_recibidas'] / perfiles['minutos_jugados']) * 90
perfiles['Pérdidas de Balón P90'] = (perfiles['perdidas_balon'] / perfiles['minutos_jugados']) * 90

# --- NUEVO: Crear la categoría general de posición ---
def agrupar_posicion(pos):
    pos = str(pos).lower()
    if 'goalkeeper' in pos: return 'Portero'
    if 'back' in pos: return 'Defensa'
    if 'midfield' in pos: return 'Mediocampista'
    if 'wing' in pos or 'forward' in pos or 'striker' in pos: return 'Delantero'
    return 'Otro'

perfiles['position_group'] = perfiles['position_detail'].apply(agrupar_posicion)

# --- MODIFICADO: Lista final con los nuevos datos cualitativos ---
columnas_finales = [
    'player', 'team', 'competition_name', 'position_group', 'position_detail',
    'minutos_jugados', 'xG P90', 'xG por Tiro', 'Asistencias a Tiro P90',
    '% Centralidad Equipo', 'Pases Progresivos P90', 'Pases Último Tercio P90',
    '% Pases Seguridad', '% Pases Bajo Presión', 'Intercepciones PAdj P90',
    'Tackles Ganados PAdj P90', 'Duelos Aéreos Ganados P90', 'Recup. Último Tercio P90',
    'Conducciones Progresivas P90', 'Influencia Global P90', 'Faltas Recibidas P90',
    'Pérdidas de Balón P90'
]

df_modelo_final = perfiles[columnas_finales].copy()

# --- NUEVO: Exportar el CSV al Drive ---
ruta_guardado = '/content/drive/MyDrive/dataset_perfiles_jugadores_1516.csv'
df_modelo_final.to_csv(ruta_guardado, index=False)

tiempo_total = round((time.time() - start_time) / 60, 2)
print(f"✅ ¡Base Maestra Lista y Guardada! Tiempo: {tiempo_total} minutos.")
print(f"Jugadores profesionales aptos (>900 min): {len(df_modelo_final)}")

🚀 INICIANDO INGENIERÍA DE BIG DATA DIRECTAMENTE DESDE LA API

⚽ Procesando La Liga...
  Partidos procesados: 380/380
⚽ Procesando Premier League...
  Partidos procesados: 380/380
⚽ Procesando Serie A...
  Partidos procesados: 380/380
⚽ Procesando Ligue 1...
  Partidos procesados: 377/377

✅ Extracción finalizada. Total partidos reales analizados: 1517
Consolidando modelo global...
✅ ¡Base Maestra Lista y Guardada! Tiempo: 29.33 minutos.
Jugadores profesionales aptos (>900 min): 1699


## 4. Corroboración Numérica Rápida

Antes de exportar el dataset consolidado, realizamos un chequeo de cordura (*sanity check*) básico. El objetivo de esta auditoría rápida no es un Análisis Exploratorio profundo (que tendrá su propio notebook dedicado), sino simplemente confirmar la integridad de los cálculos matemáticos resultantes de la ingeniería de variables.

Verificaremos dos aspectos críticos:
1.  **Ausencia de Valores Nulos (NaNs):** Para asegurar que las divisiones por 90 minutos o los cálculos de porcentajes no hayan generado errores de formato.
2.  **Rangos Lógicos:** Observar rápidamente que los valores mínimos y máximos tengan sentido dentro del contexto del fútbol (ej. que no existan promedios de 500 pases progresivos por partido o variables ofensivas negativas).

In [8]:
import pandas as pd
from IPython.display import display

print("="*85)
print(" 🔍 CORROBORACIÓN NUMÉRICA BÁSICA Y AUDITORÍA DE CEROS ABSOLUTOS")
print("="*85)

# Definir las 16 columnas analíticas
columnas_modelo = [
    'xG P90', 'xG por Tiro', 'Asistencias a Tiro P90',
    '% Centralidad Equipo', 'Pases Progresivos P90', 'Pases Último Tercio P90',
    '% Pases Seguridad', '% Pases Bajo Presión',
    'Intercepciones PAdj P90', 'Tackles Ganados PAdj P90',
    'Duelos Aéreos Ganados P90', 'Recup. Último Tercio P90',
    'Conducciones Progresivas P90', 'Influencia Global P90',
    'Faltas Recibidas P90', 'Pérdidas de Balón P90'
]

# Generar el resumen descriptivo
resumen_rapido = df_modelo_final[columnas_modelo].describe().T

# Agregar conteo de valores nulos
resumen_rapido['Valores Nulos'] = df_modelo_final[columnas_modelo].isnull().sum()

# 🚨 NUEVO: Conteo de Ceros Absolutos y su Porcentaje
total_jugadores = len(df_modelo_final)
resumen_rapido['Ceros Absolutos'] = (df_modelo_final[columnas_modelo] == 0.0).sum()
resumen_rapido['% Ceros'] = (resumen_rapido['Ceros Absolutos'] / total_jugadores) * 100

# Filtrar y ordenar las columnas para una revisión rápida
resumen_rapido = resumen_rapido[['Valores Nulos', 'Ceros Absolutos', '% Ceros', 'min', 'mean', 'max']]
resumen_rapido.columns = ['Nulos', 'Ceros (0.0)', '% Ceros', 'Mínimo', 'Promedio', 'Máximo']

# Mostrar la tabla con formato a 2 decimales
display(resumen_rapido.round(2))

print("\n✅ Corroboración finalizada. Archivo auditado y listo para el Script 2.")

 🔍 CORROBORACIÓN NUMÉRICA BÁSICA Y AUDITORÍA DE CEROS ABSOLUTOS


,Nulos,Ceros (0.0),% Ceros,Mínimo,Promedio,Máximo
xG P90,0,137,8.06,0.0,0.08,0.89
xG por Tiro,0,137,8.06,0.0,0.08,0.77
Asistencias a Tiro P90,0,114,6.71,0.0,0.55,3.57
% Centralidad Equipo,0,1,0.06,0.0,4.83,15.62
Pases Progresivos P90,0,2,0.12,0.0,12.37,35.20
Pases Último Tercio P90,0,2,0.12,0.0,10.75,41.84
% Pases Seguridad,0,16,0.94,0.0,36.42,77.78
% Pases Bajo Presión,0,4,0.24,0.0,4.36,11.14
Intercepciones PAdj P90,0,149,8.77,0.0,0.87,3.49
Tackles Ganados PAdj P90,0,131,7.71,0.0,0.80,3.23



✅ Corroboración finalizada. Archivo auditado y listo para el Script 2.


In [ ]:
# Guardar el modelo ya procesado y limpio en tu Drive
ruta_guardado = '/content/drive/MyDrive/dataset_perfiles_jugadores_1516.csv'
df_modelo_final.to_csv(ruta_guardado, index=False)

print("✅ ¡Salvado! Ya no tendrás que hacer la extracción nunca más.")